# TechCorp - Fine-tuning médical QLoRA
Ce notebook prépare 20 000 conversations, entraîne un adaptateur Phi-3.5 et l'évalue. Le modèle produit est expérimental et n'est pas destiné à un usage clinique.

## 0. Activer le GPU
Dans Colab : **Exécution > Modifier le type d'exécution > GPU**.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Activez un GPU dans Colab'

## 1. Récupérer votre version du projet
Le notebook utilise le dépôt commun de l'équipe et travaille uniquement dans rendu/ia_data.

In [ ]:
REPO_URL = 'https://github.com/Manon-Arc/Hackathon_IA.git'
!git clone {REPO_URL} Hackathon_IA
%cd Hackathon_IA/rendu/ia_data
!pip install -q -r requirements.txt

## 2. Télécharger, nettoyer et séparer le dataset médical

In [ ]:
!python scripts/prepare_datasets.py medical --max-samples 20000 --seed 42
!cat artifacts/data/medical_quality_report.md

## 3. Entraîner l'adaptateur QLoRA

In [ ]:
!python scripts/train_medical_lora.py --epochs 1 --max-length 256 --batch-size 2 --gradient-accumulation 8 --max-train-samples 1000 --max-validation-samples 200

## 4. Mesurer le modèle fine-tuné

In [ ]:
!pip uninstall -y torchao >/dev/null 2>&1 || true
!python scripts/benchmark_inference.py --max-new-tokens 96 --prompts medical_test_prompts.json --output artifacts/evaluation/medical_base.json
!python scripts/benchmark_inference.py --max-new-tokens 96 --adapter artifacts/models/phi35-medical-lora --prompts medical_test_prompts.json --output artifacts/evaluation/medical_lora.json
import json
base = json.load(open('artifacts/evaluation/medical_base.json', encoding='utf-8'))
lora = json.load(open('artifacts/evaluation/medical_lora.json', encoding='utf-8'))
{'base': base['summary'], 'lora': lora['summary']}

## 5. Archiver le livrable
Conservez l'adaptateur, le résumé d'entraînement, les rapports DATA et les résultats du benchmark.

In [ ]:
!zip -qr techcorp_ia_data_livrables.zip artifacts
from google.colab import files
files.download('techcorp_ia_data_livrables.zip')